In [0]:
-- =====================================================================================
-- SILVER TESTS (pipeline etas_silver_layer_apra, task silver_layer)
-- Silver = curated.silver_<table>, SCD Type 2 (__START_AT / __END_AT; current row = __END_AT IS NULL)
-- LoadDateTime = load time added in Silver, excluded from history tracking,
-- so reloading identical rows must NOT create new versions.
--
-- How to read DESCRIBE HISTORY (Silver):
--   MERGE            = table refreshed in this run
--   No new version   = table skipped in this run
-- =====================================================================================

-- TEST 1: refresh_group = all, first run (silver_layer only)
-- Expect: silver_current = silver_history = bronze_distinct_keys (1 row per key, no fake history)
-- Result 17 Nov 2026: PASSED (fact 124130, dim_fund 10, dim_investment_option 40, dim_asset_class 165)
SELECT 'fact' AS tbl,
  (SELECT count(*) FROM adb_alejandro_research_dev_aue_ws.curated.silver_fact_investment_strategic_allocation WHERE __END_AT IS NULL) AS silver_current,
  (SELECT count(*) FROM adb_alejandro_research_dev_aue_ws.curated.silver_fact_investment_strategic_allocation) AS silver_history,
  (SELECT count(*) FROM (SELECT DISTINCT date_key, fund_key, investment_option_key, asset_class_key FROM adb_alejandro_research_dev_aue_ws.raw_delta.bronze_fact_investment_strategic_allocation)) AS bronze_distinct_keys
UNION ALL
SELECT 'dim_fund',
  (SELECT count(*) FROM adb_alejandro_research_dev_aue_ws.curated.silver_dim_fund WHERE __END_AT IS NULL),
  (SELECT count(*) FROM adb_alejandro_research_dev_aue_ws.curated.silver_dim_fund),
  (SELECT count(DISTINCT fund_key) FROM adb_alejandro_research_dev_aue_ws.raw_delta.bronze_dim_fund)
UNION ALL
SELECT 'dim_investment_option',
  (SELECT count(*) FROM adb_alejandro_research_dev_aue_ws.curated.silver_dim_investment_option WHERE __END_AT IS NULL),
  (SELECT count(*) FROM adb_alejandro_research_dev_aue_ws.curated.silver_dim_investment_option),
  (SELECT count(DISTINCT investment_option_key) FROM adb_alejandro_research_dev_aue_ws.raw_delta.bronze_dim_investment_option)
UNION ALL
SELECT 'dim_asset_class',
  (SELECT count(*) FROM adb_alejandro_research_dev_aue_ws.curated.silver_dim_asset_class WHERE __END_AT IS NULL),
  (SELECT count(*) FROM adb_alejandro_research_dev_aue_ws.curated.silver_dim_asset_class),
  (SELECT count(DISTINCT asset_class_key) FROM adb_alejandro_research_dev_aue_ws.raw_delta.bronze_dim_asset_class);

-- =====================================================================================

-- TEST 2: refresh_group = finance_daily (bronze_layer + silver_layer), new fact csv uploaded
-- Check 1: finance table -> refreshed. Expect a NEW MERGE.
-- Result: v3 MERGE 17 Nov 02:57 UTC
DESCRIBE HISTORY adb_alejandro_research_dev_aue_ws.curated.silver_fact_investment_strategic_allocation LIMIT 3;

-- Check 2: identical rows reloaded -> no fake history. Expect both = 124130.
-- Result: 124130 / 124130 (PASSED)
SELECT count(*) AS silver_history, count_if(__END_AT IS NULL) AS silver_current
FROM adb_alejandro_research_dev_aue_ws.curated.silver_fact_investment_strategic_allocation;

-- Check 3: not finance -> skipped. Expect latest version still from the TEST 1 run (02:44 UTC).
-- Result: both still v2 02:44 UTC (PASSED)
DESCRIBE HISTORY adb_alejandro_research_dev_aue_ws.curated.silver_dim_fund LIMIT 3;
DESCRIBE HISTORY adb_alejandro_research_dev_aue_ws.curated.silver_dim_investment_option LIMIT 3;


